# 10 · Patient: LLM baselines, yes/no probes and rater agreement

**SimClinic, CS425 Group 7.** Notebook 08 (Marcus) trains and evaluates the GPT-2 patient. This notebook adds the three patient-side pieces the proposal promised that 08 does not cover:

1. **Prompted 7–8B baselines.** Section 4 of the proposal compares the fine-tuned GPT-2 models with LLaMA-3-8B-Instruct and Mistral-7B-Instruct, prompted zero-shot and few-shot, for *both* roles. Notebook 05 did the doctor; this notebook does the patient. The baselines use the same 201 test cards, the same 16 scripted questions, the same sampling (temperature 0.8, top-p 0.9) and the same metrics as notebook 08, so their rows can be added to its test table.
2. **Yes/no probes.** The scripted interview never asks about a specific symptom, so it cannot show whether a patient **agrees to symptoms it does not have**. A simulated patient that says "yes" to everything would make every doctor look good. After the 16 questions, each patient gets two probes:
   - **on-card:** "Have you had ⟨the card's last symptom⟩?" (the right answer is yes)
   - **off-card:** "Have you had ⟨a symptom of the most confusable other disease⟩?" (the right answer is no)

   The off-card symptom is chosen the way the dialogue generator chose its negatives (`generate.negative_symptoms`): a symptom of a rival disease that is not on the card, cannot be confused with a card symptom in everyday speech, and fits the persona. The probes are the same for every system.
3. **Rater agreement.** Notebook 08 averages two raters' 1–5 in-character scores. We report how far they agree (weighted κ), since an average of two raters who disagree is hard to interpret.

The GPT-2 patients get the probes too when their checkpoints are present in `models/patient/<run>/` (notebook 08's output folder). Those checkpoints were trained on the SCIS cluster and are not in the repository.

## 0. Config

In [1]:
import sys
from pathlib import Path

ROOT = Path.cwd() if (Path.cwd() / "data").exists() else Path.cwd().parent
sys.path.insert(0, str(ROOT))

LLM_MODELS = ["llama3-8b", "mistral-7b"]
STRATEGIES = ["zero_shot", "few_shot"]
GPT2_RUNS = ["stage2-gpt2-lr0.0001", "stage2-gpt2-medium-lr5e-05"]     # notebook 08's FINAL_RUNS
PATIENT_CKPT = ROOT / "models" / "patient"          # notebook 08's MODELS_DIR
TEMPERATURE, TOP_P, MAX_TOKENS = 0.8, 0.9, 120                        # notebook 08's sampling
WORKERS = 4                                                           # parallel consultations (OLLAMA_NUM_PARALLEL)
SEED = 42
RATER_CARDS = 30
CARDS_JSONL = ROOT / "data" / "processed" / "cards" / "cards.jsonl"
DIALOGUES_JSONL = ROOT / "data" / "processed" / "dialogues" / "dialogues.jsonl"
R08 = ROOT / "results run 2"                                          # notebook 08's results
OUT = ROOT / "results" / "patient"
CACHE = ROOT / "data" / "processed" / "patient_eval"
OUT.mkdir(parents=True, exist_ok=True)
CACHE.mkdir(parents=True, exist_ok=True)

## 1. Setup

In [2]:
import json
import random
import re
import time
from concurrent.futures import ThreadPoolExecutor

import numpy as np
import pandas as pd

from simclinic import cards as C
from simclinic.generate import LITERACY, TEMPERAMENT, negative_symptoms
from simclinic.lexicon import Lexicon, revealed_symptoms
from simclinic.llm import MODELS, OllamaChat
from simclinic.naive_bayes import SymcatNaiveBayes
from simclinic.symcat import Condition

pd.set_option("display.max_colwidth", 200)
pd.set_option("display.width", 200)

def read_jsonl(path):
    path = Path(path)
    return [json.loads(l) for l in path.read_text(encoding="utf-8").splitlines() if l.strip()] if path.exists() else []

all_cards = {c["card_id"]: c for c in read_jsonl(CARDS_JSONL)}
test_cards = [all_cards[c] for c in sorted(c for c, card in all_cards.items() if card["split"] == "test")]
lexicon = Lexicon.load(symptoms=sorted({s for c in all_cards.values() for s in c["symptoms"]}))
DOC = ROOT / "data" / "processed" / "doctor"
CONDITIONS = json.loads((DOC / "conditions.json").read_text())
symcat_table = {n: Condition(**c) for n, c in json.loads((DOC / "symcat_conditions.json").read_text()).items()}
nb = SymcatNaiveBayes({n: symcat_table[n].symptoms for n in CONDITIONS})
lexicon_all = Lexicon.load(symptoms=sorted({s for n in CONDITIONS for s in symcat_table[n].symptoms}))
print(len(test_cards), "test cards")

201 test cards


### Notebook 08's scoring, reproduced
The questions, the leakage name list and the score are copied from notebook 08 (`08_patient_model_job_2.ipynb`, section 5). To show that the copy is exact, the next cell re-scores 08's own test transcripts and compares the result with its saved summary.

In [3]:
QUESTIONS = [
    "What brings you in today?",
    "Where exactly do you feel it?",
    "When did it start, and did it come on suddenly or gradually?",
    "Can you describe what it feels like?",
    "Does it spread anywhere else?",
    "Have you noticed any other symptoms?",
    "Is it there all the time, or does it come and go?",
    "Does anything make it better or worse?",
    "On a scale of 0 to 10, how bad is it?",
    "Have you had anything like this before?",
    "Are you taking any medication, or have you tried anything for it?",
    "Do you have any allergies?",
    "Do you have any other medical conditions?",
    "Does anyone in your family have similar problems?",
    "Do you smoke or drink alcohol?",
    "Is there anything else you'd like to tell me?",
]

def disease_names_08(name):
    names = [name, *re.findall(r"\(([^)]+)\)", name), re.sub(r"\s*\([^)]*\)", "", name)]
    return list(dict.fromkeys([*names, *C.EXTRA_SYNONYMS.get(name, [])]))

leak_names = {d: [x for x in disease_names_08(d) if [(a, b) for a, b, _ in lexicon.find(x)] != [(0, len(x))]]
              for d in {c["disease"] for c in all_cards.values()}}

def score(card, turns):
    revealed = revealed_symptoms(turns, lexicon)
    has = [s for s, present in revealed.items() if present]
    on = [s for s in has if lexicon.on_card(s, card["symptoms"])]
    replies = [t["text"] for t in turns if t["role"] == "patient"]
    return {"mentioned": len(has), "on_card": len(on), "off_card_symptoms": "; ".join(sorted(set(has) - set(on))),
            "coverage": sum(bool(revealed.get(s)) for s in card["symptoms"]) / len(card["symptoms"]),
            "leaked": bool(C.named_diseases(" ".join(replies), leak_names[card["disease"]])),
            "empty_replies": sum(not r for r in replies),
            "reply_words": float(np.mean([len(r.split()) for r in replies]))}

def parse_transcript(text):
    turns = []
    for line in text.split("\n"):
        if line.startswith("Doctor: "):
            turns.append({"role": "doctor", "text": line[8:]})
        elif line.startswith("Patient: "):
            turns.append({"role": "patient", "text": line[9:]})
        elif turns:
            turns[-1]["text"] += "\n" + line
    return turns

def summarise(df):
    s = df.groupby("system", sort=False).agg(dialogues=("card_id", "size"), mentioned=("mentioned", "sum"),
                                              on_card=("on_card", "sum"), coverage=("coverage", "mean"),
                                              leakage=("leaked", "mean"), empty_replies=("empty_replies", "sum"),
                                              reply_words=("reply_words", "mean"))
    s.insert(3, "symptom_fidelity", s["on_card"] / s["mentioned"].replace(0, np.nan))
    return s

e08 = pd.read_csv(R08 / "patient_eval_test.csv")
rescored = pd.DataFrame([{"system": r.system, "card_id": r.card_id,
                          **score(all_cards[r.card_id], parse_transcript(r.transcript))} for r in e08.itertuples()])
mine = summarise(rescored)[["mentioned", "on_card", "leakage"]]
theirs = pd.read_csv(R08 / "patient_eval_summary_test.csv").set_index("system")[["mentioned", "on_card", "leakage"]]
check = mine.join(theirs, rsuffix=" (08)")
display(check)
assert (check["mentioned"] == check["mentioned (08)"]).all() and (check["on_card"] == check["on_card (08)"]).all(), \
    "scoring differs from notebook 08"
print("scoring reproduces notebook 08 exactly")

/tmp/ipykernel_60757/2718835675.py:62: Pandas4Warning: The 'lsuffix' and 'rsuffix' keywords in DataFrame.join are deprecated and will be removed in a future version. Use the 'suffixes' keyword instead.
  check = mine.join(theirs, rsuffix=" (08)")


,mentioned,on_card,leakage,mentioned (08),on_card (08),leakage (08)
system,,,,,,
gpt2 prompted,184,68,0.124378,184,68,0.124
gpt2-medium prompted,361,142,0.303483,361,142,0.303
stage2-gpt2-lr0.0001,633,598,0.009950,633,598,0.010
stage2-gpt2-medium-lr5e-05,672,626,0.024876,672,626,0.025
stage2-gpt2-lr5e-05-nostage1,660,602,0.004975,660,602,0.005
stage2-gpt2-medium-lr5e-05-nostage1,683,639,0.014925,683,639,0.015


scoring reproduces notebook 08 exactly


## 2. The probes
The same two probes for every system: the card's last-listed symptom (the least prominent one; the first is the presenting complaint) and the first off-card negative from `negative_symptoms` with a fixed seed per card.

In [4]:
def probes_for(card):
    rng = random.Random(f"{SEED}-{card['card_id']}")
    off = negative_symptoms(card, symcat_table, nb, lexicon_all, rng, k=1)[0]
    on = card["symptoms"][-1]
    return {"on": on, "off": off,
            "on_q": f"Have you had {lexicon_all.lay(on)}?", "off_q": f"Have you had {lexicon_all.lay(off)}?"}

PROBES = {c["card_id"]: probes_for(c) for c in test_cards}
pd.DataFrame([{"card": c["card_id"], "card symptoms": "; ".join(lexicon.lay(s) for s in c["symptoms"]),
               "on-card probe": PROBES[c["card_id"]]["on_q"], "off-card probe": PROBES[c["card_id"]]["off_q"]}
              for c in test_cards[:6]])

,card,card symptoms,on-card probe,off-card probe
0,acute-bronchitis-006,a cough; a fever; a sore throat,Have you had a sore throat?,Have you had chills?
1,acute-bronchitis-018,a cough; a blocked nose; a fever; a sore throat; shortness of breath; chest congestion,Have you had chest congestion?,Have you had ear pain?
2,acute-bronchitis-027,a cough; a blocked nose; a fever; shortness of breath; coughing up phlegm,Have you had coughing up phlegm?,Have you had body aches?
3,acute-otitis-media-008,ear pain; a fever; a cough; a blocked nose; a sore throat; reduced hearing,Have you had reduced hearing?,Have you had facial pain?
4,acute-otitis-media-010,ear pain; a fever; a blocked nose; fluid coming out of the ear; a sore throat,Have you had a sore throat?,Have you had body aches?
5,acute-otitis-media-023,ear pain; a fever; a blocked nose; fluid coming out of the ear; a sore throat; reduced hearing,Have you had reduced hearing?,Have you had coughing up phlegm?


In [5]:
UNSURE = re.compile(r"\b(?:not sure|don'?t know|do not know|unsure|not certain|can'?t say|maybe|i guess|might have)\b", re.I)

LEAD_YES = re.compile(r"^\W*(?:yes|yeah|yep|yup)\b", re.I)
LEAD_NO = re.compile(r"^\W*(?:no|nope|nah|not really|i haven't|i have not|i don't|i do not)\b", re.I)

def probe_answer(question, reply, symptom):
    """True = confirms, False = denies, None = unclear. A reply that opens with an explicit yes or no is read
    by that opening: the LLM patients often deny and then keep talking ("No, I haven't had a fever. ... I don't
    feel hot or cold"), and the lexicon can pick up a symptom from the later text. Other replies are read with
    the lexicon, as the evaluation reads everything, except that a hedged answer ("I'm not sure") is unclear:
    the lexicon reads its "not" as a no."""
    if UNSURE.search(reply) and not re.match(r"\s*(?:yes|yeah|no|nope)\b", reply, re.I):
        return None                                  # "I don't know", "maybe": unclear, not a no
    if LEAD_NO.match(reply) and not re.match(r"^\W*no,?\s+(?:but|although)\b", reply, re.I):
        return False
    if LEAD_YES.match(reply):
        return True
    return revealed_symptoms([{"role": "doctor", "text": question}, {"role": "patient", "text": reply}],
                             lexicon_all).get(symptom)

# Sanity check of the reader on hand-written answers.
for q, r, s, want in [("Have you had a fever?", "Yes, since Monday.", "Fever", True),
                      ("Have you had a fever?", "No, not really.", "Fever", False),
                      ("Have you had a fever?", "Nope.", "Fever", False),
                      ("Have you had a fever?", "I've been feverish, yes.", "Fever", True),
                      ("Have you had a fever?", "I'm not sure.", "Fever", None),
                      ("Have you had a fever?", "No, I haven't had a fever. I don't feel hot or cold.", "Fever", False),
                      ("Have you had chills?", "Yes, doctor, I've had chills.", "Chills", True)]:
    got = probe_answer(q, r, s)
    print(f"{r!r:32} -> {got}" + ("" if got == want else f"   (expected {want})"))

'Yes, since Monday.'             -> True
'No, not really.'                -> False
'Nope.'                          -> False
"I've been feverish, yes."       -> True
"I'm not sure."                  -> None
"No, I haven't had a fever. I don't feel hot or cold." -> False
"Yes, doctor, I've had chills."  -> True


## 3. The prompted LLM patients
The prompt holds the full card, as the GPT-2 header does (disease, symptoms, persona), and the same rules the dialogue generator followed: never name the disease, have only the card's symptoms, deny anything else, and stay in character. The few-shot prompt adds two training consultations of other diseases. Doctor questions are user turns and the patient's replies are assistant turns.

In [6]:
SYSTEM = """You are role-playing a patient at a general practitioner's (GP) clinic in Singapore. A doctor is interviewing you. Stay in character for the whole consultation.

Who you are: a {age}-year-old {ethnicity} {sex}, working as a {occupation}. {temperament_note} Your health literacy is {literacy}: you {literacy_note}.

Your illness (hidden from the doctor): {disease}. You do NOT know this diagnosis. Never say the name of this or any other disease, even if asked what you think it is.
Your symptoms, in your own words:
- main complaint: {main}
{others}
They started {onset} and are {severity}.

Rules:
- You have ONLY the symptoms above. If the doctor asks about any other symptom, say you do not have it.
- Mention your symptoms naturally; do not list all of them at once.
- Answer the doctor's question in 1-3 short sentences of plain speech. Write only what the patient says: no narration, no stage directions, no "Patient:" prefix."""

FEW_SHOT = "\n\nHere are two example consultations with other patients, to show the style (their illness is not yours):\n\n{examples}"

def system_prompt(card, examples=None):
    p = card["persona"]
    text = SYSTEM.format(age=p["age"], ethnicity=p["ethnicity"], sex={"F": "woman", "M": "man"}[p["sex"]],
                         occupation=p["occupation"], temperament_note=TEMPERAMENT[p["temperament"]].replace("Is ", "You are ", 1),
                         literacy=p["health_literacy"], literacy_note=LITERACY[p["health_literacy"]],
                         disease=card["disease"], main=lexicon.lay(card["symptoms"][0]),
                         others="\n".join(f"- {lexicon.lay(s)}" for s in card["symptoms"][1:]),
                         onset=p["onset"], severity=p["severity"])
    if examples:
        text += FEW_SHOT.format(examples="\n\n".join(render_example(i + 1, d) for i, d in enumerate(examples)))
    return text

def render_example(n, d):
    lines = [f"Example {n}:"] + [("Doctor: " if t["role"] == "doctor" else "Patient: ") + t["text"] for t in d["turns"][:-1]]
    return "\n".join(lines)

# Two training consultations of different diseases, 5-7 doctor questions, picked deterministically.
train = [d for d in read_jsonl(DIALOGUES_JSONL) if d["split"] == "train" and 5 <= sum(t["role"] == "doctor" for t in d["turns"]) <= 7]
train.sort(key=lambda d: d["dialogue_id"])
EXAMPLES = []
for d in train:
    if d["disease"] not in {e["disease"] for e in EXAMPLES} and all_cards[d["card_id"]]["persona"]["temperament"] not in {all_cards[e["card_id"]]["persona"]["temperament"] for e in EXAMPLES}:
        EXAMPLES.append(d)
    if len(EXAMPLES) == 2:
        break
print([(e["dialogue_id"], e["disease"]) for e in EXAMPLES])
print(system_prompt(test_cards[0], EXAMPLES)[:3000])

[('acute-bronchitis-000-d2', 'Acute bronchitis'), ('acute-otitis-media-000-d0', 'Acute otitis media')]
You are role-playing a patient at a general practitioner's (GP) clinic in Singapore. A doctor is interviewing you. Stay in character for the whole consultation.

Who you are: a 51-year-old Chinese woman, working as a retail assistant. You are chatty: adds extra detail about daily life and work. Your health literacy is low: you uses simple everyday words and may describe symptoms vaguely.

Your illness (hidden from the doctor): Acute bronchitis. You do NOT know this diagnosis. Never say the name of this or any other disease, even if asked what you think it is.
Your symptoms, in your own words:
- main complaint: a cough
- a fever
- a sore throat
They started this morning and are severe.

Rules:
- You have ONLY the symptoms above. If the doctor asks about any other symptom, say you do not have it.
- Mention your symptoms naturally; do not list all of them at once.
- Answer the doctor's q

In [7]:
def clean(reply):
    """Keep only the patient's own first turn. Mistral in particular prefixes "Me:" and then writes the
    doctor's next question and its own answer ("[[ ] Do you have any allergies? Me: ..."); everything from
    the first such continuation on is cut, as notebook 08 cuts GPT-2 at the next <|doctor|> marker."""
    reply = reply.strip()
    reply = re.sub(r"^\s*\**\s*(?:Patient|Me)\b\s*:?\s*\**\s*:?\s*", "", reply, flags=re.I)
    reply = re.sub(r"^\s*Doctor\s*:.*?(?:Me|Patient)\s*:\s*", "", reply, flags=re.I | re.S)  # echoes the question first
    reply = re.split(r"\[\[|\n\s*\n|\b(?:Doctor|Me|Patient)\s*:", reply)[0]
    reply = re.sub(r"\s*\((?:[^()]*)\)\s*", " ", reply)          # stage directions in brackets
    reply = re.sub(r"\*[^*]+\*", " ", reply)                      # *coughs*
    reply = re.sub(r"\[\s*$", "", reply.strip())
    return " ".join(reply.strip().strip('"').split())

def llm_consult(chat, card, examples, seed):
    messages = [{"role": "system", "content": system_prompt(card, examples)}]
    turns = []
    probe = PROBES[card["card_id"]]
    for i, q in enumerate(QUESTIONS + [probe["on_q"], probe["off_q"]]):
        messages.append({"role": "user", "content": q})
        reply = clean(chat(messages, max_tokens=MAX_TOKENS, temperature=TEMPERATURE, top_p=TOP_P, seed=seed + i))
        messages.append({"role": "assistant", "content": reply})
        turns += [{"role": "doctor", "text": q}, {"role": "patient", "text": reply}]
    return turns

def run_system(name, consult):
    path = CACHE / f"{re.sub(r'[^a-z0-9]+', '-', name.lower())}.jsonl"
    done = {r["card_id"] for r in read_jsonl(path)}
    todo = [c for c in test_cards if c["card_id"] not in done]
    if todo:
        t0 = time.time()
        def one(card):
            return {"system": name, "card_id": card["card_id"], "disease": card["disease"],
                    "turns": consult(card, SEED + test_cards.index(card) * 100)}
        with ThreadPoolExecutor(WORKERS) as pool, open(path, "a", encoding="utf-8") as f:
            for rec in pool.map(one, todo):
                f.write(json.dumps(rec, ensure_ascii=False) + "\n")
                f.flush()
        print(f"{name}: {len(todo)} cards in {(time.time() - t0) / 60:.1f} min")
    return read_jsonl(path)

def ollama_ready():
    import urllib.request
    try:
        tags = json.loads(urllib.request.urlopen("http://localhost:11434/api/tags", timeout=5).read())
        return {MODELS[m]["ollama"] for m in LLM_MODELS} <= {m["name"] for m in tags["models"]}
    except OSError:
        return False

records = []
if ollama_ready():
    for m in LLM_MODELS:
        chat = OllamaChat(MODELS[m]["ollama"], num_ctx=8192)
        for s in STRATEGIES:
            ex = EXAMPLES if s == "few_shot" else None
            records += run_system(f"{m} {s.replace('_', '-')}", lambda card, seed, chat=chat, ex=ex: llm_consult(chat, card, ex, seed))
        chat.unload()
else:
    print("Ollama not reachable or models missing; using cached records only")
    for p in sorted(CACHE.glob("*.jsonl")):
        records += read_jsonl(p)

## 4. GPT-2 patients on the probes
Notebook 08's final runs answer the same 16 questions and the same probes. Skipped when the checkpoints are not here.

In [8]:
import torch

def gpt2_consult(sim, card, seed):
    torch.manual_seed(seed)
    turns = []
    probe = PROBES[card["card_id"]]
    for q in QUESTIONS + [probe["on_q"], probe["off_q"]]:
        turns.append({"role": "doctor", "text": q})
        turns.append({"role": "patient", "text": sim.reply(card, turns, temperature=TEMPERATURE, top_p=TOP_P, max_new_tokens=80)})
    return turns

gpt2_available = [r for r in GPT2_RUNS if (PATIENT_CKPT / r / "config.json").exists()]
if gpt2_available:
    from simclinic.patient import PatientSimulator
    for r in gpt2_available:
        sim = PatientSimulator(str(PATIENT_CKPT / r))
        path = CACHE / f"{r}.jsonl"
        done = {x["card_id"] for x in read_jsonl(path)}
        with open(path, "a", encoding="utf-8") as f:
            for i, card in enumerate(test_cards):
                if card["card_id"] not in done:
                    f.write(json.dumps({"system": r, "card_id": card["card_id"], "disease": card["disease"],
                                        "turns": gpt2_consult(sim, card, SEED + i * 100)}, ensure_ascii=False) + "\n")
        records += read_jsonl(path)
        del sim
        torch.cuda.empty_cache()
else:
    print(f"no GPT-2 patient checkpoints in {PATIENT_CKPT.relative_to(ROOT)}/; probes run for the LLMs only")

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/292 [00:00<?, ?it/s]

## 5. Results
### 5.1 Scripted interview (same metrics as notebook 08)
Only the 16 scripted questions are scored here; the probes are scored separately below. Notebook 08's rows are its saved test results.

In [9]:
rows = []
for r in records:
    card = all_cards[r["card_id"]]
    scripted = r["turns"][: 2 * len(QUESTIONS)]
    rows.append({"system": r["system"], "card_id": r["card_id"], "disease": r["disease"], **score(card, scripted),
                 "transcript": "\n".join(("Doctor: " if t["role"] == "doctor" else "Patient: ") + t["text"] for t in scripted)})
per_dialogue = pd.DataFrame(rows)
per_dialogue.to_csv(OUT / "patient_eval_test_llm.csv", index=False)
new = summarise(per_dialogue)
old = pd.read_csv(R08 / "patient_eval_summary_test.csv").set_index("system").drop(columns=["stage2_dev_loss"], errors="ignore")
table = pd.concat([old, new.drop(index=[i for i in new.index if i in old.index])])
table = table[["dialogues", "mentioned", "on_card", "symptom_fidelity", "coverage", "leakage", "empty_replies", "reply_words"]]
table.round(3).to_csv(OUT / "patient_eval_summary_test_all.csv")
table.round(3)

,dialogues,mentioned,on_card,symptom_fidelity,coverage,leakage,empty_replies,reply_words
system,,,,,,,,
gpt2 prompted,201,184,68,0.370,0.076,0.124,0,9.093
gpt2-medium prompted,201,361,142,0.393,0.151,0.303,0,11.173
stage2-gpt2-lr0.0001,201,633,598,0.945,0.678,0.010,0,9.922
stage2-gpt2-medium-lr5e-05,201,672,626,0.932,0.713,0.025,0,10.342
stage2-gpt2-lr5e-05-nostage1,201,660,602,0.912,0.673,0.005,0,9.158
stage2-gpt2-medium-lr5e-05-nostage1,201,683,639,0.936,0.710,0.015,0,9.367
llama3-8b zero-shot,201,969,690,0.712,0.692,0.025,0,28.391
llama3-8b few-shot,201,966,709,0.734,0.715,0.015,0,27.002
mistral-7b zero-shot,201,1118,805,0.720,0.796,0.144,0,54.411


### 5.2 Bootstrap intervals
Fidelity is a pooled ratio, so the bootstrap resamples whole dialogues (cards) and recomputes the ratio.

In [10]:
all_dialogues = pd.concat([rescored.assign(source="08"), per_dialogue.assign(source="10")], ignore_index=True)
all_dialogues = all_dialogues.drop_duplicates(["system", "card_id"])
rng = np.random.default_rng(SEED)

def boot(g, n=10_000):
    m, o, leak = g["mentioned"].to_numpy(), g["on_card"].to_numpy(), g["leaked"].to_numpy(float)
    idx = rng.integers(0, len(g), (n, len(g)))
    fid = o[idx].sum(1) / np.maximum(m[idx].sum(1), 1)
    lk = leak[idx].mean(1)
    f = lambda x, full: f"{full:.3f} [{np.percentile(x, 2.5):.3f}, {np.percentile(x, 97.5):.3f}]"
    return pd.Series({"fidelity [95% CI]": f(fid, o.sum() / max(m.sum(), 1)), "leakage [95% CI]": f(lk, leak.mean())})

ci = all_dialogues.groupby("system", sort=False).apply(boot, include_groups=False)
ci.to_csv(OUT / "patient_ci_test.csv")
ci

,fidelity [95% CI],leakage [95% CI]
system,,
gpt2 prompted,"0.370 [0.291, 0.449]","0.124 [0.080, 0.169]"
gpt2-medium prompted,"0.393 [0.344, 0.444]","0.303 [0.239, 0.368]"
stage2-gpt2-lr0.0001,"0.945 [0.926, 0.962]","0.010 [0.000, 0.025]"
stage2-gpt2-medium-lr5e-05,"0.932 [0.909, 0.952]","0.025 [0.005, 0.050]"
stage2-gpt2-lr5e-05-nostage1,"0.912 [0.890, 0.934]","0.005 [0.000, 0.015]"
stage2-gpt2-medium-lr5e-05-nostage1,"0.936 [0.915, 0.955]","0.015 [0.000, 0.035]"
llama3-8b zero-shot,"0.712 [0.682, 0.742]","0.025 [0.005, 0.050]"
llama3-8b few-shot,"0.734 [0.703, 0.765]","0.015 [0.000, 0.035]"
mistral-7b zero-shot,"0.720 [0.683, 0.756]","0.144 [0.100, 0.194]"


### 5.3 Probes
- **confirms on-card:** says yes when asked about a symptom it has (higher is better)
- **confirms off-card:** says yes to a symptom it does *not* have (lower is better: this is the "yes to everything" failure)
- **unclear:** the answer is neither a clear yes nor a clear no to the lexicon

In [11]:
probe_rows = []
for r in records:
    p = PROBES[r["card_id"]]
    (on_q, on_r), (off_q, off_r) = [(r["turns"][i]["text"], r["turns"][i + 1]["text"]) for i in (-4, -2)]
    assert on_q == p["on_q"] and off_q == p["off_q"]
    probe_rows.append({"system": r["system"], "card_id": r["card_id"], "on_answer": probe_answer(on_q, on_r, p["on"]),
                       "off_answer": probe_answer(off_q, off_r, p["off"]), "on_reply": on_r, "off_reply": off_r,
                       "on_q": on_q, "off_q": off_q})
probes = pd.DataFrame(probe_rows)
probes.to_csv(OUT / "patient_probes_test.csv", index=False)

def probe_summary(g):
    return pd.Series({"cards": len(g),
                      "confirms on-card": (g["on_answer"] == True).mean(),
                      "denies on-card": (g["on_answer"] == False).mean(),
                      "confirms off-card": (g["off_answer"] == True).mean(),
                      "denies off-card": (g["off_answer"] == False).mean(),
                      "unclear (either)": ((g["on_answer"].isna()) | (g["off_answer"].isna())).mean()})

probe_table = probes.groupby("system", sort=False).apply(probe_summary, include_groups=False).round(3)
probe_table.to_csv(OUT / "patient_probes_summary_test.csv")
probe_table

,cards,confirms on-card,denies on-card,confirms off-card,denies off-card,unclear (either)
system,,,,,,
llama3-8b zero-shot,201.0,0.990,0.000,0.119,0.861,0.030
llama3-8b few-shot,201.0,0.990,0.000,0.139,0.856,0.015
mistral-7b zero-shot,201.0,1.000,0.000,0.139,0.861,0.000
mistral-7b few-shot,201.0,0.990,0.010,0.129,0.866,0.005
stage2-gpt2-lr0.0001,201.0,0.945,0.055,0.025,0.975,0.000
stage2-gpt2-medium-lr5e-05,201.0,0.677,0.308,0.060,0.935,0.015


**Manual audit of the GPT-2 patients' on-card denials (read on 2026-11-06).** GPT-2 medium "denies" its own last card symptom on 62 cards (31%). It often opens with "No," and then confirms: "No, my throat feels like it's really sore when I swallow." or "No, it's just a sore on my lip." It also seems to read "Have you had X?" as "have you ever had X before?" ("No, I've never had that before"). Reading all 62:

| GPT-2 medium, on-card probe answered "no" | cards |
|---|---|
| confirms despite the opening "No" | 17 |
| ambiguous ("never had that before", incoherent) | 11 |
| real denial of a card symptom ("No, I haven't had a fever.") | 34 (17% of 201) |

GPT-2 small "denies" on 11 cards. About 5 of those confirm in the same way, so its real denial rate is about 3%. We read every off-card confirmation of both GPT-2 patients and they are real ("Have you had chills?" "Yeah, yeah."). **So the off-card rates hold: 2.5% (small) and 6% (medium), against 12–14% for the LLM patients. The medium patient is the less reliable of the two under direct yes/no questions: it denies a card symptom in about 17–22% of cases.** Under the doctors' own history-taking questions in notebook 12, the same model reveals 64% of card symptoms with fidelity 0.96, so the problem is specific to this question form.


In [12]:
# Examples of off-card confirmations, the failure the probe looks for.
probes[probes["off_answer"] == True][["system", "card_id", "off_q", "off_reply"]].groupby("system").head(3)

,system,card_id,off_q,off_reply
0,llama3-8b zero-shot,acute-bronchitis-006,Have you had chills?,"Yes, I have had chills. It's been a weird feeling, like my whole body is shivering and I'm cold, even when I'm sitting in a warm room."
22,llama3-8b zero-shot,asthma-005,Have you had difficulty breathing?,"Yeah, I have trouble breathing sometimes. It's like my chest is tight and I can't get a full breath in. It's really scary, to be honest."
25,llama3-8b zero-shot,athlete-s-foot-000,Have you had itchy skin?,"Yes, doctor. I have itchy skin sometimes, especially during the dry season. But this is different. This rash is itchy and painful, and it's not just itchy skin. I'm worried, doctor, that it might ..."
201,llama3-8b few-shot,acute-bronchitis-006,Have you had chills?,"Yeah! I've had chills too. It's like my whole body is cold, even though I'm wearing a sweater."
210,llama3-8b few-shot,acute-sinusitis-032,Have you had shortness of breath?,"Sometimes, yes. When I'm coughing really badly, it can be hard to catch my breath. It's like my lungs are all congested and it's hard to get air in. It's really scary and uncomfortable."
211,llama3-8b few-shot,anxiety-005,Have you had sleepiness?,"Yeah, I've been feeling really tired all the time. Even when I do get some sleep, I still feel exhausted."
402,mistral-7b zero-shot,acute-bronchitis-006,Have you had chills?,"Yes, doctor, I've had chills. The fever makes me feel hot and shivery at the same time. It comes and goes, but it's been a constant presence since this morning. And it's made it hard for me to get..."
424,mistral-7b zero-shot,asthma-005,Have you had difficulty breathing?,"Yes, sometimes it feels like I can't get enough air and I have to take deep breaths to try and catch my breath. I'm really worried, is this a sign of something serious?[ ][ ][ ][ ][ ][ ][ ][ ][ ][..."
427,mistral-7b zero-shot,athlete-s-foot-000,Have you had itchy skin?,"Yes, doc. The rash on my foot has been really itchy, especially at night. It's hard for me to sleep because of the itching and the pain. I'm really worried and scared about what it could be. Is it..."
603,mistral-7b few-shot,acute-bronchitis-006,Have you had chills?,"Yes, I've had chills, doctor. I've been feeling really cold and shivery, especially when I have a fever. I've been wrapping myself in blankets and trying to stay warm, but it doesn't seem to be ma..."


## 6. Rater agreement (notebook 08's sheets)
Two raters scored the dev sheet 1–5. We report exact and within-one agreement, quadratic-weighted Cohen's κ, and each rater's mean per system. The test sheet is reported once it has scores.

In [13]:
from sklearn.metrics import cohen_kappa_score
from scipy.stats import spearmanr

def agreement(split):
    sheet, key = R08 / f"rater_sheet_{split}.csv", R08 / f"rater_key_{split}.csv"
    s = pd.read_csv(sheet).merge(pd.read_csv(key), on="sheet_id").dropna(subset=["rater_1", "rater_2"])
    if s.empty:
        print(f"{split}: no scores yet")
        return None, None
    a, b = s["rater_1"].astype(int), s["rater_2"].astype(int)
    overall = pd.Series({"transcripts": len(s), "exact agreement": (a == b).mean(), "within 1 point": ((a - b).abs() <= 1).mean(),
                         "quadratic-weighted kappa": cohen_kappa_score(a, b, weights="quadratic"),
                         "unweighted kappa": cohen_kappa_score(a, b), "Spearman rho": spearmanr(a, b).statistic,
                         "rater 1 mean": a.mean(), "rater 2 mean": b.mean()}).round(3)
    per_system = s.groupby("system")[["rater_1", "rater_2"]].mean().assign(mean=lambda d: d.mean(axis=1)).round(2)
    print(f"{split}: score distribution rater 1 {a.value_counts().sort_index().to_dict()}, rater 2 {b.value_counts().sort_index().to_dict()}")
    return overall, per_system

dev_agree, dev_per_system = agreement("dev")
test_agree, test_per_system = agreement("test")
display(dev_agree.to_frame("dev"))
display(dev_per_system)
dev_agree.to_frame("dev").to_csv(OUT / "rater_agreement_dev.csv")

dev: score distribution rater 1 {2: 14, 3: 19, 4: 5, 5: 62}, rater 2 {2: 5, 3: 25, 4: 61, 5: 9}
test: no scores yet


,dev
transcripts,100.000
exact agreement,0.140
within 1 point,0.630
quadratic-weighted kappa,-0.142
unweighted kappa,-0.001
Spearman rho,-0.174
rater 1 mean,4.150
rater 2 mean,3.740


,rater_1,rater_2,mean
system,,,
stage2-gpt2-lr0.0001,4.02,3.58,3.80
stage2-gpt2-medium-lr5e-05,4.28,3.90,4.09


### 6.1 A supplementary rater sheet for the LLM patients
08's test sheet covers prompted GPT-2 and the two fine-tuned runs. So the in-character scores also cover the 7–8B baselines, we write a second, blinded sheet in the same format: the best LLM configuration of each model (by symptom fidelity) on the same 30 cards 08 sampled. Raters fill it in with the same rubric. It is not overwritten once it has scores.

In [14]:
sheet_path, key_path = OUT / "rater_sheet_test_llm.csv", OUT / "rater_key_test_llm.csv"
cards_08 = pd.read_csv(R08 / "rater_key_test.csv")["card_id"].unique().tolist()
if not per_dialogue.empty:
    best = (new.assign(model=[i.split()[0] for i in new.index]).sort_values("symptom_fidelity", ascending=False)
               .groupby("model").head(1).index.tolist())
    rated = per_dialogue[per_dialogue["system"].isin(best) & per_dialogue["card_id"].isin(cards_08)]

    def describe(card):
        p = ", ".join(f"{k.replace('_', ' ')}: {v}" for k, v in card["persona"].items())
        return f"{card['disease']} | symptoms: {'; '.join(lexicon.lay(s) for s in card['symptoms'])} | {p}"

    if sheet_path.exists() and pd.read_csv(sheet_path)[["rater_1", "rater_2"]].notna().any().any():
        print(f"{sheet_path.name} already has scores, not overwritten")
    else:
        order = rated.sample(frac=1, random_state=SEED).reset_index(drop=True)
        order["sheet_id"] = [f"L{i:04d}" for i in range(len(order))]
        pd.DataFrame({"sheet_id": order["sheet_id"], "card": order["card_id"].map(lambda c: describe(all_cards[c])),
                      "transcript": order["transcript"], "rater_1": np.nan, "rater_2": np.nan, "notes": ""}).to_csv(sheet_path, index=False)
        order[["sheet_id", "system", "card_id"]].to_csv(key_path, index=False)
        print(f"wrote {sheet_path.relative_to(ROOT)}: {len(order)} transcripts from {best}")

wrote results/patient/rater_sheet_test_llm.csv: 120 transcripts from ['stage2-gpt2-lr0.0001', 'stage2-gpt2-medium-lr5e-05', 'mistral-7b few-shot', 'llama3-8b few-shot']


## 7. Example transcripts
One card, every system, for the report's qualitative section.

In [15]:
example_card = test_cards[0]["card_id"]
for name, g in all_dialogues[all_dialogues["card_id"] == example_card].groupby("system", sort=False):
    tr = g.iloc[0].get("transcript")
    if not isinstance(tr, str):
        tr = e08[(e08["system"] == name) & (e08["card_id"] == example_card)]["transcript"].iloc[0]
    print(f"=== {name}\n" + "\n".join(tr.split("\n")[:10]) + "\n")

=== gpt2 prompted
Doctor: What brings you in today?
Patient: It's my friend and I have a little bit of a problem with asthma, I have a little bit of asthma. I get asthma when I get outside and I don't know how I got there, and then I get my first bronchitis.
Doctor: Where exactly do you feel it?
Patient: I'm a little bit on the bad side of asthma, but the problem is I have a really bad cough. It's really bad, and I think I'm on the good side, so I just think I'm getting a bad cough, and then I have a little bit of an allergy, and then I have an allergy to certain medicines.
Doctor: When did it start, and did it come on suddenly or gradually?
Patient: It's been pretty long since I've seen it.
Doctor: Can you describe what it feels like?
Patient: It's like a burning sensation. I don't know, it feels like burning.
Doctor: Does it spread anywhere else?
Patient: It doesn't spread anywhere.

=== gpt2-medium prompted
Doctor: What brings you in today?
Patient: It's my friend's birthday.
Doctor